# Full Pipeline Debug Notebook

Run the KKBox churn workflow end to end: ingestion, feature engineering, preprocessing, training, and artifact checks.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "config" / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import get_path, get_value, load_config

CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
config = load_config(CONFIG_PATH)
INTERIM_DIR = get_path(config, "interim_dir", base_dir=PROJECT_ROOT)
PROCESSED_DIR = get_path(config, "processed_dir", base_dir=PROJECT_ROOT)
REPORTS_DIR = get_path(config, "reports_dir", base_dir=PROJECT_ROOT)
MODELS_DIR = get_path(config, "models_dir", base_dir=PROJECT_ROOT)
preview_rows = int(get_value(config, "eda", "preview_rows"))
print(f"Project root: {PROJECT_ROOT}")

## 1. Ingestion

Load the raw CSVs through the ingestion stage and write compact interim Parquet outputs.

In [ ]:
from src.data.run_ingestion import main as run_ingestion_stage

run_ingestion_stage()

ingestion_outputs = [
    "train.parquet",
    "members.parquet",
    "transactions_summary.parquet",
    "user_logs_summary.parquet",
    "modeling_frame.parquet",
]
for name in ingestion_outputs:
    path = INTERIM_DIR / name
    print(f"{name}: {path.exists()} - {path}")

## 2. Feature Engineering

Build the leak-safe feature frame and save the metadata summary.

In [ ]:
from src.features.pipeline import run_feature_engineering

feature_frame = run_feature_engineering(CONFIG_PATH)
print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head(preview_rows))

## 3. Preprocessing

Create train/validation/test splits and persist the tree-model preprocessor.

In [ ]:
from src.features.preprocess import run_preprocessing

splits = run_preprocessing(config, PROJECT_ROOT)
for name in ["X_train.parquet", "X_val.parquet", "X_test.parquet", "y_train.parquet", "y_val.parquet", "y_test.parquet"]:
    path = PROCESSED_DIR / name
    print(f"{name}: {path.exists()} - {path}")

## 4. Training

Train all candidate models, select the champion, and save comparison artifacts.

In [ ]:
from src.models.train import run_training

champion, all_results, comparison_df = run_training(config, PROJECT_ROOT)
print("Champion model:", champion.name)
display(comparison_df)

## 5. Artifact Summary

Check the saved comparison table and interpretation outputs.

In [ ]:
report_files = [
    REPORTS_DIR / "model_comparison.csv",
    REPORTS_DIR / "model_comparison_topk.csv",
    REPORTS_DIR / "model_comparison_optuna.csv",
    REPORTS_DIR / "feature_importances.csv",
    REPORTS_DIR / "shap_summary.csv",
    REPORTS_DIR / "shap_summary_sample.csv",
]
model_files = [
    MODELS_DIR / "champion_model.pkl",
    MODELS_DIR / "champion_name.txt",
    MODELS_DIR / "preprocessor.pkl",
]

status = pd.DataFrame({"artifact": [str(path) for path in report_files + model_files], "exists": [path.exists() for path in report_files + model_files]})
display(status)